In [35]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.multiclass import OneVsOneClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [36]:
from sklearn.model_selection import train_test_split

'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)


In [37]:
def check_bias_variance(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

    accuracy_diff = train_accuracy - test_accuracy

    if abs(accuracy_diff) > 0.05:  # 5% threshold
        if train_accuracy > test_accuracy:
            print("Potential high variance (overfitting)\n")
        else:
            print("Potential high bias (underfitting)\n")

    elif train_accuracy < 0.7 and test_accuracy < 0.7:  # 70% threshold
        print("Model is balanced but has low overall accuracy (potential underfitting)\n")
    else:
        print("Model is balanced\n")


In [38]:
# Linear

cv_results = cross_validate(OneVsOneClassifier(SVC(kernel='linear')), X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier LINEAR : ')
check_bias_variance(train_accuracy, test_accuracy)

OneVsOneClassifier LINEAR : 
Training Accuracy: 58.848%
Testing Accuracy: 57.618%
Model is balanced but has low overall accuracy (potential underfitting)



In [39]:
ovo_model = OneVsOneClassifier(SVC(kernel='linear'))

param_grid_linear = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}


grid_search = GridSearchCV(ovo_model, param_grid_linear, cv=5, scoring='accuracy', verbose=2)

# Fit the grid search model
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print("Test Set Accuracy:", test_accuracy)


Fitting 5 folds for each of 7 candidates, totalling 35 fits
[CV] END .................................estimator__C=0.001; total time=   0.0s
[CV] END .................................estimator__C=0.001; total time=   0.0s
[CV] END .................................estimator__C=0.001; total time=   0.0s
[CV] END .................................estimator__C=0.001; total time=   0.0s
[CV] END .................................estimator__C=0.001; total time=   0.0s
[CV] END ..................................estimator__C=0.01; total time=   0.0s
[CV] END ..................................estimator__C=0.01; total time=   0.0s
[CV] END ..................................estimator__C=0.01; total time=   0.0s
[CV] END ..................................estimator__C=0.01; total time=   0.0s
[CV] END ..................................estimator__C=0.01; total time=   0.0s
[CV] END ...................................estimator__C=0.1; total time=   0.0s
[CV] END ...................................estim